In [1]:
import os
import numpy as np
import random
import torch
import pandas as pd
from torch import nn, optim
import random
from PIL import Image
import timm
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from sklearn.linear_model import LogisticRegression

In [2]:
def set_random_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = "42"
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

set_random_seed()

In [3]:
class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, transform1=None, transform2=None, img_dir='/kaggle/input/datasets/yuulind/pa-100k/PA-100K/data/'):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform1 = transform1
        self.transform2 = transform2
        
    def __len__(self):
        return len(self.img_labels)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = Image.open(img_path).convert('RGB')
        label = torch.from_numpy(self.img_labels.iloc[idx, 1:].values.astype(np.float32))
        
        
        image1 = self.transform1(image)    
        image2 = self.transform2(image)
        
        return image1, image2, label

In [4]:
eff_transform = transforms.Compose([
    transforms.Resize((384, 192), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor()
])


swin_transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor()
])

test_dataset = CustomImageDataset('/kaggle/input/datasets/yuulind/pa-100k/PA-100K/test.csv', eff_transform, swin_transform)
val_dataset = CustomImageDataset('/kaggle/input/datasets/yuulind/pa-100k/PA-100K/val.csv', eff_transform, swin_transform)

In [5]:
# Common DataLoader kwargs
loader_kwargs = {
    'batch_size': 16,
    'num_workers': 4,
    'pin_memory': True,
    'persistent_workers': True,
}


val_loader = DataLoader(
    val_dataset, 
    shuffle=False,
    **loader_kwargs
)

test_loader = DataLoader(
    test_dataset, 
    shuffle=False,
    **loader_kwargs
)

In [6]:
class CSRA(nn.Module): 
    def __init__(self, input_dim, num_classes, lam):
        super(CSRA, self).__init__()          
        self.lam = lam
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(input_dim, 256, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.SiLU())
        
        self.conv3 = nn.Sequential(
            nn.Conv2d(input_dim, 256, 3, bias=False, padding=1),
            nn.BatchNorm2d(256),
            nn.SiLU())
        
        self.conv5 = nn.Sequential(
            nn.Conv2d(input_dim, 256, 5, bias=False, padding=2),
            nn.BatchNorm2d(256),
            nn.SiLU())
        
        self.head = nn.Conv2d(256*3, num_classes, 1, bias=False)
        self.softmax = nn.Softmax(dim=2)

    def forward(self, x):
        feat1 = self.conv1(x)
        feat3 = self.conv3(x)
        feat5 = self.conv5(x)
        multi_feat = torch.cat([feat1, feat3, feat5], dim=1)

        score = self.head(multi_feat) / torch.norm(self.head.weight, dim=1, keepdim=True).transpose(0,1)
        score = score.flatten(2)
        base_logit = torch.mean(score, dim=2)
        att_logit = torch.max(score, dim=2)[0]

        return base_logit + self.lam * att_logit


class Eff_CSRA(nn.Module):
    def __init__(self, lam, num_classes):
        super(Eff_CSRA, self).__init__()
        self.backbone = models.efficientnet_v2_m(weights='DEFAULT').features
        self.classifier = CSRA(1280, num_classes, lam)
        
    def forward(self, x):
        return self.classifier(self.backbone(x))

class Swin_CSRA(nn.Module):
    def __init__(self, lam, num_classes):
        super(Swin_CSRA, self).__init__()
        self.backbone = timm.create_model("swin_small_patch4_window7_224", pretrained=True, features_only=True)
        self.classifier = CSRA(768, num_classes, lam)
        
    def forward(self, x):
        return self.classifier(self.backbone(x)[-1].permute(0,3, 1, 2))

In [7]:
def compute_mFive(y_true, y_pred):
    y_true = np.array(y_true, dtype='float32')
    y_pred = np.array(y_pred, dtype='float32')
    
    # ---- mA (mean per-label accuracy) ----
    L = y_true.shape[1]
    TP = np.sum((y_true == 1) & (y_pred == 1), axis=0)
    TN = np.sum((y_true == 0) & (y_pred == 0), axis=0)
    P = np.sum(y_true == 1, axis=0)
    N = np.sum(y_true == 0, axis=0)

    tp_over_p = np.where(P == 0, 1.0, TP / P)
    tn_over_n = np.where(N == 0, 1.0, TN / N)

    mA = (1 / (2 * L)) * np.sum(tp_over_p + tn_over_n)

    # ---- Instance-level metrics ----
    intersection = np.logical_and(y_true, y_pred).sum(axis=1)
    union = np.logical_or(y_true, y_pred).sum(axis=1)
    pred_sum = y_pred.sum(axis=1)
    true_sum = y_true.sum(axis=1)

    eps = 1e-8  # small number to avoid division by zero
    Accuracy = np.mean(intersection / (union + eps))
    Precision = np.mean(intersection / (pred_sum + eps))
    Recall = np.mean(intersection / (true_sum + eps))
    F1 = 2 * Precision * Recall / (Precision + Recall)

    # ---- mFive ----
    mFive = np.mean([mA, Accuracy, Precision, Recall, F1])

    return {
        'mA': mA,
        'Accuracy': Accuracy,
        'Precision': Precision,
        'Recall': Recall,
        'F1': F1,
        'mFive': mFive
    }

In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eff_model=Eff_CSRA(0.05, num_classes=26).to(device)
swin_model=Swin_CSRA(0.03, num_classes=26).to(device)
eff_weights = torch.load('/kaggle/input/pk/eff_augmented_best_weights_0.05.pth')
swin_weights = torch.load('/kaggle/input/pk/swin_augmented_best_weights_0.03.pth')

eff_model.load_state_dict(eff_weights)
swin_model.load_state_dict(swin_weights)


swin_model.eval()
eff_model.eval()
print("Model weights are loaded")

Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth
100%|██████████| 208M/208M [00:01<00:00, 178MB/s]  


model.safetensors:   0%|          | 0.00/200M [00:00<?, ?B/s]

Model weights are loaded


In [9]:
def get_predictions(model1, model2, loader):
    """Get predictions from both models (probabilities)"""
    all_probs1 = []
    all_probs2 = []
    all_labels = []
    
    with torch.no_grad():
        for img1, img2, labels in tqdm(loader, desc="Getting predictions"):
            img1 = img1.to(device)
            img2 = img2.to(device)
            
            logits1 = model1(img1)
            logits2 = model2(img2)
            
            probs1 = torch.sigmoid(logits1)
            probs2 = torch.sigmoid(logits2)
            
            all_probs1.append(probs1.cpu().numpy())
            all_probs2.append(probs2.cpu().numpy())
            all_labels.append(labels.numpy())
    
    return (np.vstack(all_probs1), np.vstack(all_probs2), np.vstack(all_labels))

In [10]:
# Get predictions on validation set
print("Getting validation predictions...")
val_probs1, val_probs2, val_labels = get_predictions(eff_model, swin_model, val_loader)

# Get predictions on test set
print("\nGetting test predictions...")
test_probs1, test_probs2, test_labels = get_predictions(eff_model, swin_model, test_loader)

Getting validation predictions...


Getting predictions: 100%|██████████| 625/625 [01:42<00:00,  6.08it/s]



Getting test predictions...


Getting predictions: 100%|██████████| 625/625 [01:41<00:00,  6.15it/s]


In [11]:
# 1. Simple Averaging
simple_probs = (test_probs1 + test_probs2) / 2.0
simple_preds = (simple_probs >= 0.5).astype(int)
simple_average_ensemble_metrics = compute_mFive(test_labels, simple_preds)
print("Simple Averaging Ensemble on Test Set:")
simple_average_ensemble_metrics

Simple Averaging Ensemble on Test Set:


{'mA': 0.8551172143048886,
 'Accuracy': 0.8257385664207048,
 'Precision': 0.8860234920634921,
 'Recall': 0.9068350396825398,
 'F1': 0.8963084753849124,
 'mFive': 0.8740045575713074}

In [12]:
# 2. Weighted Averaging
best_weight = 0.5
best_mfive = 0
for w in np.arange(0, 1.05, 0.05):
    weighted_probs = w * val_probs1 + (1 - w) * val_probs2
    weighted_preds = (weighted_probs >= 0.5).astype(int)
    metrics = compute_mFive(val_labels, weighted_preds)
    if metrics['mFive'] > best_mfive:
        best_mfive = metrics['mFive']
        best_weight = w


print(f"Optimal weight: eff_model={best_weight:.3f}, swin_model={1-best_weight:.3f}")

weighted_probs = (best_weight * test_probs1 + (1 - best_weight) * test_probs2) 
weighted_preds = (weighted_probs >= 0.5).astype(int)
weighted_average_ensemble_metrics = compute_mFive(test_labels, weighted_preds)

print(f"Weighted Averaging Ensemble on Test Set:")
weighted_average_ensemble_metrics

Optimal weight: eff_model=0.500, swin_model=0.500
Weighted Averaging Ensemble on Test Set:


{'mA': 0.8551172143048886,
 'Accuracy': 0.8257385664207048,
 'Precision': 0.8860234920634921,
 'Recall': 0.9068350396825398,
 'F1': 0.8963084753849124,
 'mFive': 0.8740045575713074}

In [13]:
# Step 1: Prepare meta-features (predictions from base models)
X_val_meta = np.column_stack([val_probs1, val_probs2])  # Shape: (n_val, 70)
y_val_meta = val_labels  # Shape: (n_val, 35)

# Prepare test meta-features once (outside the loop for efficiency)
X_test_meta = np.column_stack([test_probs1, test_probs2])  # Shape: (n_test, 70)

# Initialize test predictions array
test_predictions = np.zeros_like(test_probs1)  # Shape: (n_test, 35)


print(f"\nTraining stacking on validation set...")

# For each label (binary classification), train a meta-learner
for label_idx in range(26):
    print(f" Training label {label_idx+1}/26...", end='\r')
    y_label = y_val_meta[:, label_idx]
    
    # Train meta-learner on validition data
    meta = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    meta.fit(X_val_meta, y_label)
    
    # Predict on test set
    X_test_meta = np.column_stack([test_probs1, test_probs2])
    test_predictions[:, label_idx] = meta.predict_proba(X_test_meta)[:, 1]

print("Stacking training completed")


test_binary = (test_predictions >= 0.5).astype(int)

stacking_ensemble_metrics = compute_mFive(test_labels, test_binary)

print(f"Stacking Ensemble on Test Set:")
stacking_ensemble_metrics


Training stacking on validation set...
Stacking training completed
Stacking Ensemble on Test Set:


{'mA': 0.8070397800480835,
 'Accuracy': 0.8268886421381718,
 'Precision': 0.9115405158730159,
 'Recall': 0.8820625,
 'F1': 0.8965592711059163,
 'mFive': 0.8648181418330374}